# Available data exploration

**Purpose.** Take stock of the datasets already downloaded for the thesis,
answer basic descriptive questions about them, and identify which data are
still missing to answer RQ1–RQ3 (see `docs/research_questions.md`). **RQ4
(temporal mismatch) is deferred from the current scope as of 2026-10-17** —
not discarded, just postponed; see `docs/research_questions.md` and
`docs/research_gaps.md` (Scope and Feasibility). Some analysis below (the
reference-year coverage check) still matters for RQ1–RQ3's dataset
construction even with RQ4 deferred.

**Scope.** Exploratory only — no modelling, no filtering decisions. All data
here are *pilot downloads* for access validation (see `data/README.md`), not
a species/region decision.

| Dataset | Location | Covered here |
|---|---|---|
| iNaturalist *Acacia*, Coimbra district | `data/raw/citizen_science/` | Section 2 |
| TESSERA embeddings 2024, Coimbra municipality | `data/raw/satellite/tessera_coimbra_pilot/` | Section 3 |
| COS 2023 (national land cover) | `data/raw/satellite/cos/` | not explored in this notebook |

Questions for the citizen-science data:

1. How many observations are there?
2. What dates do they have / what is the temporal period?
3. Where are they located?
4. Are there several observations at the same place?
5. Are there areas with many observations and others with almost none?
6. What information comes with each observation?
7. Is there any indication of quality/confidence?

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from geotessera import dequantize_embedding
from shapely.geometry import box

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
INAT_CSV = RAW / "citizen_science" / "inaturalist_acacia_coimbra_pilot.csv"
TESSERA_DIR = RAW / "satellite" / "tessera_coimbra_pilot"
TESSERA_YEAR = 2024

# CRS used for metric computations (distances, grid cells). It is the CRS the
# TESSERA tiles are delivered in, so observations and embeddings line up.
UTM = "EPSG:32629"

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110

## 1. Inventory of downloaded files

In [ ]:
rows = []
for f in sorted(RAW.rglob("*")):
    if f.is_file() and f.name != ".gitkeep":
        rel = f.relative_to(RAW)
        rows.append({"group": "/".join(rel.parts[:2]), "file": f.name, "size_MB": f.stat().st_size / 1e6})
inventory = pd.DataFrame(rows)
inventory.groupby("group").agg(n_files=("file", "size"), size_MB=("size_MB", "sum")).round(1)

## 2. Citizen science — iNaturalist *Acacia* (Coimbra district pilot)

### 2.1 What information comes with each observation? (Q1, Q6)

In [ ]:
inat = pd.read_csv(INAT_CSV)
inat["observed_on"] = pd.to_datetime(inat["observed_on"], errors="coerce")
print(f"{len(inat):,} observations, {inat['id'].nunique():,} unique ids, {inat.shape[1]} columns")
inat.head()

In [ ]:
pd.DataFrame({
    "dtype": inat.dtypes.astype(str),
    "n_missing": inat.isna().sum(),
    "pct_missing": (inat.isna().mean() * 100).round(1),
    "n_unique": inat.nunique(),
    "example": inat.iloc[0],
})

Notes on the columns (from the iNaturalist API, as documented in `data/README.md`):

- `quality_grade` — iNaturalist's own data-quality label (`research`, `needs_id`, `casual`).
- `positional_accuracy` — observer-reported coordinate uncertainty, in metres. Missing for most records — check below.
- `coordinates_obscured` — true when the location was deliberately randomised (privacy or threatened taxon).
- `time_observed_at` — time of day; often missing, and not needed at TESSERA's annual resolution.
- There is **no abundance, cover, or absence information**: each row is a presence-only record.

### 2.2 Temporal period and date distribution (Q2)

In [ ]:
dates = inat["observed_on"]
print("missing dates:", dates.isna().sum())
print("earliest:", dates.min().date(), "| latest:", dates.max().date())

year_counts = dates.dt.year.value_counts().sort_index()
year_share = (year_counts / year_counts.sum() * 100).round(1)
pd.DataFrame({"n": year_counts, "pct": year_share}).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
yc = year_counts[year_counts.index >= 2010]
axes[0].bar(yc.index.astype(int), yc.values, color="tab:green")
axes[0].axvspan(2016.5, 2025.5, color="grey", alpha=0.15, label="TESSERA availability (2017–2025)")
axes[0].axvline(TESSERA_YEAR, color="black", ls="--", lw=1, label="TESSERA year downloaded")
axes[0].set_title("Observations per year (>= 2010)")
axes[0].legend(fontsize=8)

mc = dates.dt.month.value_counts().sort_index()
axes[1].bar(mc.index.astype(int), mc.values, color="tab:green")
axes[1].set_xticks(range(1, 13))
axes[1].set_title("Observations per month (all years)")
plt.tight_layout()

How much of the data is temporally close to a given reference year? **RQ4
(a dedicated temporal-mismatch experiment) is deferred** (supervisor
decision, 2026-10-17 — see `docs/research_questions.md`), but RQ1–RQ3 still
need *some* fixed rule for pairing an observation with a satellite year —
this table is the evidence for that choice (see `docs/research_gaps.md`,
Scope and Feasibility, options 1–2). `TESSERA_YEAR` is the year actually
downloaded; 2023 is included because OrtoSat and COS are fixed at 2023.

In [ ]:
years = dates.dt.year
coverage = {}
for ref in [2023, TESSERA_YEAR]:
    coverage[ref] = {f"±{k}": round(((years - ref).abs() <= k).mean() * 100, 1) for k in range(4)}
coverage["TESSERA 2017–2025 (nearest year)"] = {"±0": round(years.between(2017, 2025).mean() * 100, 1)}
pd.DataFrame(coverage).T.rename_axis("reference year")

### 2.3 Where are the observations? (Q3)

Two pilot boundaries are in play (see `data/README.md`): the iNaturalist query
used the Coimbra **district**, the TESSERA download covers 12 tiles of 0.1°
around the Coimbra **municipality**. The tile names give the tile centre, so
the downloaded tiles span lon −8.6 to −8.3 and lat 40.0 to 40.4.

In [ ]:
inat_gdf = gpd.GeoDataFrame(
    inat, geometry=gpd.points_from_xy(inat["longitude"], inat["latitude"]), crs="EPSG:4326"
).to_crs(UTM)

tile_dirs = sorted((TESSERA_DIR / "global_0.1_degree_representation" / str(TESSERA_YEAR)).iterdir())
tile_centres = [tuple(float(v) for v in d.name.removeprefix("grid_").split("_")) for d in tile_dirs]
tiles = gpd.GeoDataFrame(
    {"tile": [d.name for d in tile_dirs]},
    geometry=[box(lon - 0.05, lat - 0.05, lon + 0.05, lat + 0.05) for lon, lat in tile_centres],
    crs="EPSG:4326",
).to_crs(UTM)

tessera_area = tiles.union_all()
inat_gdf["in_tessera"] = inat_gdf.within(tessera_area)
print(f"Observations inside downloaded TESSERA tiles: {inat_gdf['in_tessera'].sum():,} "
      f"of {len(inat_gdf):,} ({inat_gdf['in_tessera'].mean():.1%})")
inat[["latitude", "longitude"]].describe().loc[["min", "max", "mean"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
tiles.boundary.plot(ax=ax, color="black", lw=0.6)
for q, colour in [("research", "tab:green"), ("needs_id", "tab:orange"), ("casual", "tab:grey")]:
    sub = inat_gdf[inat_gdf["quality_grade"] == q]
    sub.plot(ax=ax, markersize=2, color=colour, alpha=0.5, label=f"{q} (n={len(sub):,})")
ax.set_title("iNaturalist Acacia observations; black grid = downloaded TESSERA tiles")
ax.legend(markerscale=5, fontsize=8)
ax.set_xlabel("Easting (m, UTM 29N)"); ax.set_ylabel("Northing (m)")
plt.tight_layout()

### 2.4 Several observations at the same place? (Q4)

Checked at increasing tolerance: identical coordinates, then the same 10 m
cell (= one TESSERA / Sentinel-2 pixel), 100 m and 1 km cells. Many records
sharing a pixel means they will collapse to the same feature vector once
satellite data are attached — relevant for dataset construction and for
spatial leakage between train and test.

In [ ]:
def grid_cell(gdf, size_m):
    return (gdf.geometry.x // size_m).astype(int).astype(str) + "_" + (gdf.geometry.y // size_m).astype(int).astype(str)

rows = [{
    "unit": "identical coordinates",
    "n_locations": inat.groupby(["latitude", "longitude"]).ngroups,
    "max_obs_per_location": inat.groupby(["latitude", "longitude"]).size().max(),
}]
for size in [10, 100, 1000]:
    counts = grid_cell(inat_gdf, size).value_counts()
    rows.append({"unit": f"{size} m cell", "n_locations": counts.size, "max_obs_per_location": counts.max()})
dup = pd.DataFrame(rows)
dup["obs_per_location_mean"] = (len(inat) / dup["n_locations"]).round(2)
dup["pct_obs_sharing_location"] = [
    round(inat.duplicated(["latitude", "longitude"], keep=False).mean() * 100, 1),
    *[round(grid_cell(inat_gdf, s).duplicated(keep=False).mean() * 100, 1) for s in [10, 100, 1000]],
]
dup

In [ ]:
same_place_same_day = inat.groupby(["latitude", "longitude", "observed_on"]).size()
print("Locations with >1 record on the same day (exact coords):", (same_place_same_day > 1).sum())
print("Records involved:", same_place_same_day[same_place_same_day > 1].sum())

### 2.5 Uneven sampling: dense areas vs. empty areas (Q5)

Counts per 1 km cell over the bounding box of the observations. The Lorenz
curve shows how concentrated observations are among the *occupied* cells; the
share of empty cells shows how much of the area has no record at all
(which must **not** be read as absence of *Acacia*).

In [ ]:
cell = 1000
xmin, ymin, xmax, ymax = inat_gdf.total_bounds
nx, ny = int(np.ceil((xmax - xmin) / cell)), int(np.ceil((ymax - ymin) / cell))
H, xedges, yedges = np.histogram2d(inat_gdf.geometry.x, inat_gdf.geometry.y, bins=[nx, ny],
                                   range=[[xmin, xmin + nx * cell], [ymin, ymin + ny * cell]])
occupied = np.sort(H[H > 0])[::-1]
print(f"1 km cells in bbox: {H.size:,} | with >=1 observation: {occupied.size:,} ({occupied.size / H.size:.1%})")
for k in [1, 10, 50]:
    print(f"Top {k:>2} cells hold {occupied[:k].sum() / occupied.sum():.1%} of observations")

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.6, 1]})
im = axes[0].imshow(np.ma.masked_equal(H.T, 0), origin="lower", cmap="viridis",
                    norm=plt.matplotlib.colors.LogNorm(),
                    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]])
tiles.boundary.plot(ax=axes[0], color="red", lw=0.5)
fig.colorbar(im, ax=axes[0], label="observations per km² (log)")
axes[0].set_title("Observation density, 1 km cells (empty cells blank)")

asc = np.sort(occupied)
axes[1].plot(np.linspace(0, 1, asc.size + 1), np.r_[0, asc.cumsum() / asc.sum()], color="tab:green")
axes[1].plot([0, 1], [0, 1], ls="--", color="grey", lw=1)
axes[1].set_xlabel("cumulative share of occupied cells"); axes[1].set_ylabel("cumulative share of observations")
axes[1].set_title("Lorenz curve (occupied 1 km cells)")
plt.tight_layout()

### 2.6 Quality / confidence indicators (Q7)

In [ ]:
summary = pd.concat({
    "quality_grade": inat["quality_grade"].value_counts(),
    "taxon_rank": inat["taxon_rank"].value_counts(),
    "coordinates_obscured": inat["coordinates_obscured"].value_counts(),
}).to_frame("n")
summary["pct"] = (summary["n"] / len(inat) * 100).round(1)
summary

In [ ]:
acc = inat["positional_accuracy"]
print(f"positional_accuracy missing: {acc.isna().mean():.1%}")
bins = [0, 10, 30, 100, 1000, np.inf]
labels = ["<=10 m (within a pixel)", "10–30 m", "30–100 m", "100 m–1 km", ">1 km"]
pd.cut(acc.dropna(), bins=bins, labels=labels, include_lowest=True).value_counts().sort_index().to_frame("n")

Taxonomy: the genus query returns several species with different ecology
(and probably different spectral/phenological signatures). Records at genus
or subgenus level cannot be assigned to a species.

In [ ]:
species = inat.groupby("taxon_name").agg(
    n=("id", "size"),
    rank=("taxon_rank", "first"),
    pct_research=("quality_grade", lambda q: round((q == "research").mean() * 100, 1)),
).sort_values("n", ascending=False)
species["pct_of_total"] = (species["n"] / len(inat) * 100).round(1)
species

### 2.7 Observer concentration

Who produced the data matters for bias (RQ3): a single very active observer
imprints their own routes and habits on the spatial and temporal distribution.

In [ ]:
obs_per_user = inat["user_login"].value_counts()
print(f"{obs_per_user.size} distinct observers")
print(f"Top observer: {obs_per_user.iloc[0]:,} records ({obs_per_user.iloc[0] / len(inat):.1%})")
print(f"Top 10 observers: {obs_per_user.iloc[:10].sum() / len(inat):.1%}")
print(f"Observers with a single record: {(obs_per_user == 1).sum()}")

top_user = obs_per_user.index[0]
inat["observer_group"] = np.where(inat["user_login"] == top_user, "top observer", "all others")
inat_gdf["observer_group"] = inat["observer_group"].values
pd.crosstab(inat["observer_group"], inat["quality_grade"], margins=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pd.crosstab(dates.dt.year, inat["observer_group"]).loc[2015:].plot.bar(stacked=True, ax=axes[0], width=0.8)
axes[0].set_title("Observations per year by observer group")
tiles.boundary.plot(ax=axes[1], color="black", lw=0.5)
for grp, colour in [("all others", "tab:blue"), ("top observer", "tab:orange")]:
    inat_gdf[inat_gdf["observer_group"] == grp].plot(ax=axes[1], markersize=2, alpha=0.5, color=colour, label=grp)
axes[1].legend(markerscale=5); axes[1].set_title("Spatial footprint by observer group")
plt.tight_layout()

### 2.8 Licensing

In [ ]:
inat["license_code"].value_counts(dropna=False).to_frame("n")

## 3. Satellite — TESSERA embeddings (Coimbra municipality, 2024)

Per `data/README.md`: 12 tiles, quantised `int8` embeddings (128 channels) +
per-pixel scale factors, with a georeferenced landmask GeoTIFF per tile. The
`.npy` files carry no georeference — the matching GeoTIFF provides the CRS and
transform.

In [ ]:
TIFF_DIR = TESSERA_DIR / "global_0.1_degree_tiff_all"
tile_rows = []
for d in tile_dirs:
    emb = np.load(d / f"{d.name}.npy", mmap_mode="r")
    scales = np.load(d / f"{d.name}_scales.npy", mmap_mode="r")
    with rasterio.open(TIFF_DIR / f"{d.name}.tiff") as src:
        mask = src.read(1)
        tile_rows.append({
            "tile": d.name, "emb_shape": emb.shape, "emb_dtype": str(emb.dtype),
            "scales_shape": scales.shape, "tiff_shape": src.shape, "crs": src.crs.to_string(),
            "pixel_m": src.res[0], "shapes_match": emb.shape[:2] == src.shape,
            "pct_land": round((mask == 1).mean() * 100, 1),
        })
tessera_tiles = pd.DataFrame(tile_rows)
tessera_tiles

Dequantisation follows `geotessera.dequantize_embedding` (embedding × per-pixel
scale). Below: value range and a false-colour view of one tile (first 3
principal components of the 128-d embedding, fitted on a pixel sample) to
check the data look spatially coherent.

In [ ]:
demo = tile_dirs[len(tile_dirs) // 2]
emb = dequantize_embedding(np.load(demo / f"{demo.name}.npy"), np.load(demo / f"{demo.name}_scales.npy"))
print(demo.name, emb.shape, emb.dtype)
print("value range:", float(emb.min()), "to", float(emb.max()), "| NaN:", int(np.isnan(emb).sum()))

flat = emb.reshape(-1, emb.shape[-1])
rng = np.random.default_rng(0)
sample = flat[rng.choice(flat.shape[0], 20_000, replace=False)]
mean = sample.mean(axis=0)
_, _, vt = np.linalg.svd(sample - mean, full_matrices=False)
pcs = ((flat - mean) @ vt[:3].T).reshape(*emb.shape[:2], 3)
lo, hi = np.percentile(pcs, [2, 98], axis=(0, 1))
rgb = np.clip((pcs - lo) / (hi - lo), 0, 1)

plt.figure(figsize=(6, 7))
plt.imshow(rgb)
plt.title(f"{demo.name} — embedding PCs 1–3 as RGB")
plt.axis("off");

### 3.1 Can the observations be linked to the embeddings?

Sample the 128-d embedding at each observation inside the downloaded tiles
(nearest pixel). This is a feasibility check, not dataset construction: no
filtering, no temporal matching (all records get the 2024 embedding regardless
of their observation year).

In [ ]:
def sample_tile(tile_dir, points):
    with rasterio.open(TIFF_DIR / f"{tile_dir.name}.tiff") as src:
        rows, cols = rasterio.transform.rowcol(src.transform, points.geometry.x, points.geometry.y)
        h, w = src.shape
    rows, cols = np.asarray(rows), np.asarray(cols)
    ok = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
    emb = np.load(tile_dir / f"{tile_dir.name}.npy", mmap_mode="r")
    scales = np.load(tile_dir / f"{tile_dir.name}_scales.npy", mmap_mode="r")
    vecs = emb[rows[ok], cols[ok]].astype(np.float32) * scales[rows[ok], cols[ok]][:, None]
    return points.index[ok], vecs

pts = inat_gdf[inat_gdf["in_tessera"]]
tile_of_point = gpd.sjoin(pts[["geometry"]], tiles, predicate="within")["tile"]
idx, vecs = [], []
for d in tile_dirs:
    sub = pts.loc[tile_of_point.index[tile_of_point == d.name]]
    if len(sub):
        i, v = sample_tile(d, sub)
        idx.extend(i); vecs.append(v)
point_embeddings = pd.DataFrame(np.vstack(vecs), index=idx).sort_index()
print(f"Embeddings sampled for {len(point_embeddings):,} observations "
      f"({point_embeddings.round(4).drop_duplicates().shape[0]:,} distinct vectors — duplicates = same pixel)")
print("Observation years of these records:")
print(dates.loc[point_embeddings.index].dt.year.value_counts().sort_index().to_dict())

## 4. Summary of findings

The numbers below are computed above; the interpretation is mine and should
be checked against the data and the supervisor's view. Labels follow
`CLAUDE.md` (**Established fact** / **Reasonable proposal** / **Open question**).

In [ ]:
facts = {
    "iNat observations": len(inat),
    "date range": f"{dates.min().date()} – {dates.max().date()}",
    "share from top year": f"{year_share.max()}% ({int(year_share.idxmax())})",
    "share research grade": f"{(inat['quality_grade'] == 'research').mean():.1%}",
    "share from top observer": f"{obs_per_user.iloc[0] / len(inat):.1%}",
    "positional_accuracy missing": f"{acc.isna().mean():.1%}",
    "distinct 10 m pixels": dup.loc[dup['unit'] == '10 m cell', 'n_locations'].item(),
    "obs inside TESSERA tiles": f"{inat_gdf['in_tessera'].sum()} ({inat_gdf['in_tessera'].mean():.1%})",
    "TESSERA years downloaded": TESSERA_YEAR,
}
pd.Series(facts, name="value").to_frame()

**Established facts (from this notebook)**

- The pilot is **presence-only**: no absence, abundance or cover information.
- The temporal distribution is very uneven (2021 alone is the largest year;
  little data before 2019).
- A **single observer** produced the large majority of records, and almost
  all `casual` records — so quality grade and observer identity are
  confounded. Spatial and temporal patterns largely reflect that observer.
  This single-region, single-observer-dominated pilot cannot yet support
  the geographic comparison RQ3 calls for (see below).
- Many records share a 10 m pixel, and a small number of 1 km cells hold a
  large share of the data — strong spatial clustering.
- Only a minority of observations fall inside the downloaded TESSERA tiles,
  because the two pilots use different boundaries. That subset is also
  temporally different from the whole: mostly 2024–2026 rather than the
  2021 peak — so any conclusion drawn on it may not transfer to the rest.
- `positional_accuracy` is missing for most records, so coordinate error
  relative to the 10 m pixel cannot be assessed for most of the data.

**Open questions raised**

- Should `casual` records be used at all? (Need to look up *why* they are
  casual — on iNaturalist this can mean e.g. no photo, captive/cultivated, or
  missing date; the CSV does not say which.)
- How to deal with the dominant observer (down-weight, thin, or study it as a
  bias source for RQ3)?
- Species-level vs. genus-level target (*A. dealbata* dominates; *A. longifolia*
  — the species with a Portuguese Sentinel-2 precedent — is much rarer here).
- **RQ2 (supervisor refinement, 2026-10-17):** reduce data via a
  distribution-improving filter rather than random percentages, applied to
  a densely-observed area, and record which observations it removes and
  why. Section 2.5's density map (dense 1 km cells vs. the Lorenz curve) is
  a candidate area to design and test that filter on, but the filter itself
  is not yet defined or implemented here.
- **RQ4 is deferred, not answered** — the reference-year coverage table in
  Section 2.2 is reused below to pick the RQ1–RQ3 default, but no RQ4
  mismatch experiment is planned for now.

## 5. What data are still missing (vs. RQ1–RQ3; RQ4 deferred)

| Need | Why (RQ) | Current status | Label |
|---|---|---|---|
| **Conventional Sentinel-2 (and possibly Sentinel-1) features** over the study area | RQ1, RQ2 baseline — the comparison is impossible without it | Not obtained (needs Copernicus Data Space / GEE account) | Established fact (gap) |
| **Aligned study area** — TESSERA for the whole district, or iNat clipped to the municipality | All RQs; most observations currently have no embedding | Boundaries not reconciled | Established fact (gap) |
| **A second, geographically distinct region** (same or comparable species) | RQ3 — the supervisor-confirmed comparison is *across* geographically distinct areas; this single-region pilot cannot provide that by itself | Not obtained | Established fact (gap) |
| **A defined distribution-improving filter for RQ2** (criterion for which observations to remove, in a densely-observed area) | RQ2, per supervisor refinement 2026-10-17 | Not defined or implemented | Open question |
| **Negative / background samples** | Any classifier needs a negative class; presence-only data provide none | No strategy, no data | Open question |
| **Independent validation reference** (not from the same citizen-science source) | Evaluation of all RQs; RQ3 in particular, since biased training data evaluated on equally biased test data hides the bias effect | None | Open question |
| **Bias covariates** — roads/trails, accessibility (e.g. OpenStreetMap), population | RQ3, per Dimson et al. (2023) bias axes in `research_gaps.md` | Not obtained | Reasonable proposal |
| **Study-area boundary** (CAOP administrative limits) | Clipping, area statistics, mapping | Not obtained (currently using bounding boxes) | Reasonable proposal |
| **More citizen-science records** — other iNat years/regions, or GBIF (which also aggregates other sources) | RQ2 needs enough data for the filtering approach above; RQ3 needs a second region, not just spatial variety within one | Only one pilot query | Open question |
| **AlphaEarth (or another) foundation model** | RQ1 if more than one FM is compared | Not obtained, access unconfirmed | Open question |
| **COS 2023** | Land-cover context, possible negatives/validation | Downloaded; exploration deferred | — |
| ~~TESSERA for other years (2017–2025 available)~~ | ~~RQ4~~ — lower priority now RQ4 is deferred; still useful if nearest-year matching (Section 2.2) is chosen as the RQ1–RQ3 default | Only 2024 downloaded | Reasonable proposal (lower priority) |